<div style="display: flex; align-items: center; justify-content: flex-start; text-align: left;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/6/68/Logo_universidad_icesi.svg/960px-Logo_universidad_icesi.svg.png" width="300" style="margin-right: 20px;">
    <div>
        <h3 style="margin: 0;">FACULTAD BARBERI DE INGENIERÍA, DISEÑO Y CIENCIAS APLICADAS</h3>
        <h3 style="margin: 0;">ALGORITMOS Y PROGRAMACIÓN III</h3>
    </div>
</div>

# Proyecto Final

Integrantes:
- ANGY MARIA HURTADO OSORIO A00401755
- HIDEKI TAMURA HERNANDEZ A00348618
- DAVID VERGARA LAVERDE A00402237


## Modelado - Experimentos de Machine Learning Tradicional

Este notebook contiene los experimentos utilizando modelos clásicos (Random Forest, SVM, XGBoost) para predecir **simultáneamente** la calidad y el tamaño de las frutas/verduras usando un enfoque `MultiOutputClassifier`.

Siguiendo las mejores prácticas, las funciones de evaluación pesadas se han abstraído en `src/evaluation/evaluate.py`.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

# Añadir la carpeta raíz al path para poder importar desde la carpeta src
sys.path.append(os.path.abspath('..'))

from src.evaluation.evaluate import evaluate_multioutput_model

### 1. Carga de Datos y Preprocesamiento

Para el Machine Learning tradicional, asumimos que las características (features) ya han sido extraídas en la fase de preparación de datos (ej. histogramas de color, descriptores de textura HOG, contornos). 

Aquí utilizaremos datos simulados representativos para mantener la estructura funcional. *Debes reemplazar esto con la carga de tus datos reales cuando estén listos.*

In [ ]:
# TODO: Reemplazar con la carga real de características usando src.utils.helpers
# X = np.load('../data/processed/features.npy')
# Y = pd.read_csv('../data/processed/labels.csv')

# Generando datos simulados para validación de la pipeline
np.random.seed(42)
num_samples = 500
num_features = 20 # Supongamos 20 características extraídas (ej. color, textura)

X = np.random.rand(num_samples, num_features)

# Y_quality: 0 (Mala), 1 (Regular), 2 (Buena)
# Y_size: 0 (Pequeño), 1 (Mediano), 2 (Grande)
y_quality = np.random.choice([0, 1, 2], size=num_samples)
y_size = np.random.choice([0, 1, 2], size=num_samples)

# Agrupamos las etiquetas para el MultiOutputClassifier
Y = np.column_stack((y_quality, y_size))

# División en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f'Dimensiones X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'Dimensiones X_test: {X_test.shape}, y_test: {y_test.shape}')

### 2. Definición del Modelo Base: Random Forest con MultiOutputClassifier

Utilizamos `MultiOutputClassifier` para permitir que los modelos clásicos (como Random Forest o SVM) predigan ambas variables objetivo (Calidad y Tamaño) al mismo tiempo.

In [ ]:
rf_base = RandomForestClassifier(random_state=42)
multi_target_rf = MultiOutputClassifier(rf_base, n_jobs=-1)

# Pipeline para escalar datos (importante para SVM, menos crítico para RF pero buena práctica)
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', multi_target_rf)
])

# Entrenar un modelo base inicial
pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)

### 3. Evaluación del Modelo Base

Llamamos a nuestra función modular centralizada para evaluar el rendimiento, evitando ensuciar el notebook con código repetitivo de gráficas y reportes.

In [ ]:
quality_names = ['Mala', 'Regular', 'Buena']
size_names = ['Pequeño', 'Mediano', 'Grande']

# y_test[:, 0] es calidad, y_test[:, 1] es tamaño
evaluate_multioutput_model(y_test[:, 0], y_pred_rf[:, 0],
                           y_test[:, 1], y_pred_rf[:, 1],
                           quality_classes=quality_names,
                           size_classes=size_names)

### 4. Ajuste de Hiperparámetros (Hyperparameter Tuning)

Aplicamos `GridSearchCV` para evitar el overfitting y encontrar la mejor combinación de parámetros. 

**Nota:** al usar `MultiOutputClassifier` y `Pipeline`, debemos acceder a los parámetros del modelo base especificando la ruta completa en el diccionario de parámetros: `classifier__estimator__<parametro>`.

In [ ]:
param_grid = {
    'classifier__estimator__n_estimators': [50, 100, 150],
    'classifier__estimator__max_depth': [None, 10, 20],
    'classifier__estimator__min_samples_split': [2, 5]
}

# GridSearchCV con Cross-Validation de 5 pliegues para evitar overfitting
grid_search = GridSearchCV(pipeline_rf, param_grid, cv=5, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f'Mejores parámetros encontrados: {grid_search.best_params_}')
best_rf_model = grid_search.best_estimator_

### 5. Evaluación del Mejor Modelo Encontrado

In [ ]:
y_pred_best = best_rf_model.predict(X_test)

evaluate_multioutput_model(y_test[:, 0], y_pred_best[:, 0],
                           y_test[:, 1], y_pred_best[:, 1],
                           quality_classes=quality_names,
                           size_classes=size_names)

### 6. Guardar el Modelo Entrenado

Guardamos el mejor modelo utilizando `joblib` para poder consumirlo posteriormente en la interfaz de Streamlit (`app.py`).

In [ ]:
model_path = '../models/best_rf_multioutput.pkl'
os.makedirs('../models', exist_ok=True)

joblib.dump(best_rf_model, model_path)
print(f'Modelo guardado exitosamente en: {model_path}')